In [ ]:
!pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
!pip install  sentencepiece protobuf datasets huggingface_hub hf_transfer
!pip install --no-deps unsloth

In [ ]:
from unsloth import FastVisionModel
import torch

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-2B-Instruct-unsloth-bnb-4bit",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth"
)

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,

    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state = 3407,
    use_rslora=False,
    loftq_config=None
)

In [ ]:
from datasets import load_dataset
dataset = load_dataset("naver-clova-ix/cord-v2", split="train[:500]")

In [ ]:
dataset

In [ ]:
dataset[0]

In [ ]:
dataset[0]["image"]

In [ ]:
dataset[0]['ground_truth']

In [ ]:
dataset[2]["image"]


In [ ]:
instruction = "Extract the vendor_name, date, and total_amount from this receipt. Return ONLY JSON."

In [ ]:
def format_data(sample):
    content = [
            {"role": "user", "content": [{"type": "image", "image": sample["image"]}, {"type": "text", "text": instruction}]},
            {"role": "assistant", "content": [{"type": "text", "text": sample["ground_truth"]}]}
        ]

    return {"messages":content}

In [ ]:
format_data(dataset[0])

In [ ]:
formatted_data = [format_data(sample) for sample in dataset]


In [ ]:
formatted_data[1]

In [ ]:
FastVisionModel.for_inference(model)

In [ ]:
image = dataset[1]["image"]
messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": instruction},
            {"type": "image", "image": image}
        ]
    }
]

In [ ]:
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(
    image, input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

In [ ]:
import sys
import torch

# 1. System Safety
sys.setrecursionlimit(5000)
FastVisionModel.for_inference(model)

# 2. Hardware Preparation
torch.cuda.empty_cache()
inputs = {k: v.to("cuda") for k, v in inputs.items()}

# 3. Precise Extraction
with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens = 128,
        use_cache = True,
        temperature = 0.01, # Set extremely low for "Exact" JSON extraction
        pad_token_id = tokenizer.pad_token_id,
        eos_token_id = tokenizer.eos_token_id,
    )

generated_text = tokenizer.decode(output_ids[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)
print(f"\nFinal Extracted JSON:\n{generated_text}")

In [ ]:
image

In [ ]:
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

In [ ]:
FastVisionModel.for_inference(model)

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer=tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = formatted_data,
    args = SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps = 5,
        max_steps=30,
        learning_rate=2e-4,
        fp16=not is_bf16_supported(),
        bf16 = is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to = "none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs = {"skip_prepare_dataset": True},
        dataset_num_proc=4,
        max_seq_length=2048,
    ),
)

In [ ]:
trainer.train()


In [ ]:
FastVisionModel.for_inference(model)


In [ ]:
image = dataset[2]["image"]


In [ ]:
instruction = "Extract the vendor_name, date and total_amount (receipt ID only if its available) from this receipt. Return ONLY JSON."

In [ ]:
messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]

In [ ]:
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(
    image, input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

In [ ]:
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

_ = model.generate(**inputs, streamer=text_streamer, max_new_tokens=128, use_cache=True, temperature=1.5, min_p=0.1)

In [ ]:
image

In [ ]:
# Save the LoRA adapters to a local folder
model.save_pretrained("qwen3_adapters")
tokenizer.save_pretrained("qwen3_adapters")

# Optional: Zip the folder so you can download it to your PC or upload to GitHub
!zip -r receipt_adapters.zip qwen3_adapters

In [ ]:
# Run this cell to load the model and add the fine-tuned weights to the qwen3 vllm without retraining again.

from unsloth import FastVisionModel
import torch

# 1. Load the BASE model (The original library)
model, tokenizer = FastVisionModel.from_pretrained(
    model_name = "unsloth/Qwen3-VL-2B-Instruct-unsloth-bnb-4bit",
    load_in_4bit = True,
)

# 2. "Plug in" your saved Brain (The adapters)
model = FastVisionModel.get_peft_model(
    model,
    model_name = "receipt_adapters", # Point to your saved folder
)

# 3. Set to Inference Mode
FastVisionModel.for_inference(model)

print("Model reloaded and ready for extraction!")

In [ ]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI, UploadFile, File
import gradio as gr
import os, json, torch
from PIL import Image
import io

# 1. Background execution & Inference Mode
nest_asyncio.apply()
# from unsloth import FastVisionModel
FastVisionModel.for_inference(model)

app = FastAPI()
DB_FILE = "receipt_database.json"

# --- CORE LOGIC FUNCTIONS ---

def perform_inference(image):
    # Standard extraction prompt
    messages = [
        {"role": "user", "content": [
            {"type": "text", "text": "Extract vendor_name, date, and total_amount. Return ONLY JSON."},
            {"type": "image", "image": image}
        ]}
    ]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")

    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=128, use_cache=True)

    return tokenizer.decode(output_ids[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)

def grounded_qa(query):
    if not os.path.exists(DB_FILE):
        return "No receipts uploaded yet. Please upload a receipt in the first tab."

    with open(DB_FILE, "r") as f:
        context = json.load(f)

    # SYSTEM PROMPT for grounding
    qa_prompt = f"You are a helpful assistant. Based ONLY on this JSON data: {json.dumps(context)}, answer the question: {query}"

    # For Chat, we treat it as a text-only task, but Qwen3-VL still needs the chat template
    messages = [{"role": "user", "content": [{"type": "text", "text": qa_prompt}]}]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

    # We pass None for image because the chat doesn't need a specific new image
    inputs = tokenizer(None, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")

    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=128, use_cache=True)

    return tokenizer.decode(output_ids[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)

# --- UI WRAPPERS ---

def process_and_save(img):
    if img is None: return "Please upload an image first."

    raw_output = perform_inference(img)
    # Robust cleaning of markdown backticks
    clean_json = raw_output.replace("```json", "").replace("```", "").strip()

    db = []
    if os.path.exists(DB_FILE):
        try:
            with open(DB_FILE, "r") as f: db = json.load(f)
        except: db = []

    try:
        data_dict = json.loads(clean_json)
        db.append(data_dict)
        with open(DB_FILE, "w") as f: json.dump(db, f, indent=4)
        return clean_json
    except Exception as e:
        return f"Extraction successful, but JSON parsing failed. Model said: {raw_output}"

# --- GRADIO UI ---

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# DocEx and QAsys")

    with gr.Tabs():
        with gr.TabItem("📤 Upload & Extract"):
            with gr.Row():
                img_input = gr.Image(type="pil", label="Step 1: Upload Receipt")
                json_output = gr.Code(label="Step 2: Extracted Data", language="json")
            btn = gr.Button("Extract and Save to Database", variant="primary")
            btn.click(fn=process_and_save, inputs=img_input, outputs=json_output)

        with gr.TabItem("💬 Chat with Data"):
            chatbot = gr.Chatbot(label="Receipt Assistant")
            msg = gr.Textbox(label="Ask a question about your saved receipts")

            def chat_wrapper(query, history):
                response = grounded_qa(query)
                return history + [[query, response]]

            msg.submit(fn=chat_wrapper, inputs=[msg, chatbot], outputs=chatbot)

# --- LAUNCH ---
app = gr.mount_gradio_app(app, demo, path="/")
demo.launch(share=True, inline=True)